# Support Ops Env — Gemma 4 GRPO Training

**OpenEnv Hackathon Submission** | [HF Space](https://huggingface.co/spaces/raj23211/support-ops-env) | [GitHub](https://github.com/raj23211/support-ops-env)

Train `google/gemma-4-E2B-it` on the Support Ops Env using **GRPO** with HF TRL's `environment_factory` API. Gemma 4 E2B is post-trained for multi-turn tool calling — Tau2 (average) score **24.5%** zero-shot vs **16.2%** for Gemma 3 27B — so GRPO has a real signal to climb from.

| Component | Detail |
|-----------|--------|
| Environment | [HF Space](https://huggingface.co/spaces/raj23211/support-ops-env) (deterministic grader) |
| Model       | `google/gemma-4-E2B-it` (5.1B total, 2.3B effective) |
| Quantization | 4-bit (bitsandbytes) — fits Colab T4 16GB |
| Framework   | TRL git HEAD + Transformers ≥4.57 |

The training signal is generated **at rollout time** by the agent interacting with the environment. No pre-collected dataset.

## 1. Install Dependencies

Gemma 4 requires TRL **git HEAD** (for `environment_factory` + `max_tool_calling_iterations` + `chat_template_kwargs`) and **Transformers 5.5.4+**. The install cell pins `5.5.4`, the first stable release that recognizes `model_type=gemma4`.

In [ ]:
# Support Ops Env install — Gemma 4 path.
# Order matters:  pin matched torch family first, then install everything else.
# This avoids torch<->torchvision drift when bitsandbytes forces torch down to 2.9.x.

# 1) Matched torch 2.9.1 family (what bitsandbytes wants, matching torchvision/audio).
!pip install -q --no-cache-dir \
    torch==2.9.1 torchvision==0.24.1 torchaudio==2.9.1 \
    --index-url https://download.pytorch.org/whl/cu128

# 2) ML stack (TRL git HEAD required for environment_factory + max_tool_calling_iterations).
!pip install -q --no-cache-dir "trl @ git+https://github.com/huggingface/trl.git"
!pip install -q --no-cache-dir "transformers==5.5.4" accelerate peft datasets matplotlib bitsandbytes jmespath

# 3) openenv-core runtime dep (our env package's only hard dep besides the stdlib).
!pip install -q --no-cache-dir "openenv-core[core]>=0.2.3"

# 4) Our env package WITHOUT --gemma extras (torch etc. are already set above).
!pip install -q --no-cache-dir --no-deps \
    "openenv-support-ops @ git+https://huggingface.co/spaces/raj23211/support-ops-env@main"

# 5) Verify the matched torch family survived the install.
import torch, torchvision
assert torch.__version__.startswith("2.9"),        f"torch got changed to {torch.__version__}"
assert torchvision.__version__.startswith("0.24"), f"torchvision got changed to {torchvision.__version__}"
print(f"torch       : {torch.__version__}  ✓")
print(f"torchvision : {torchvision.__version__}  ✓")
print("\n--- install OK. RESTART RUNTIME before running Section 2. ---")

## 2. Configuration

Add `HF_TOKEN` to Colab Secrets (key icon). Thinking mode is **ON** by default — that is what Tau2 tests, and Gemma 4 E2B's headline scores come from thinking enabled.

In [ ]:
import os

# Try Colab Secrets first; fall back to env var so this cell works anywhere.
try:
    from google.colab import userdata  # type: ignore
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    if "HF_TOKEN" not in os.environ:
        print("WARNING: HF_TOKEN not set. Either:")
        print("  - Click the key icon in the Colab sidebar, add HF_TOKEN, enable notebook access, re-run this cell.")
        print("  - Or run: os.environ['HF_TOKEN'] = 'hf_...'")

ENV_URL         = "https://raj23211-support-ops-env.hf.space"
MODEL_ID        = "google/gemma-4-E2B-it"
HUB_REPO        = "raj23211/support-ops-agent-gemma4-e2b"
NUM_EPISODES    = 50
NUM_GENERATIONS = 4           # free T4 comfortably handles 4
MAX_TOOL_ITERS  = 12          # max tool calls per episode
MAX_COMPLETION_LEN = 1024     # raise to 1536 if enabling thinking
ENABLE_THINKING = True        # Tau2 scores use thinking=True

print(f"Environment : {ENV_URL}")
print(f"Model       : {MODEL_ID}")
print(f"Episodes    : {NUM_EPISODES}")
print(f"Thinking    : {ENABLE_THINKING}")

## 3. Smoke Test — Verify Environment Connectivity

Connect to the HF Space, reset, and call one tool to confirm round-trip.

In [ ]:
from support_ops_env import SupportOpsAction, SupportOpsEnv, TASK_IDS

print(f"Connecting to {ENV_URL} ...")

env = SupportOpsEnv(base_url=ENV_URL).sync()   # EnvClient is async; .sync() gives a blocking adapter
try:
    reset_result = env.reset(task_id=TASK_IDS[0])
    obs = reset_result.observation
    print("Connected.\n")
    print(f"Task       : {obs.task_title}")
    print(f"Collection : {obs.collection}")
    print(f"Objective  : {obs.objective[:200]}...\n")

    step = env.step(
        SupportOpsAction(
            assistant_message="smoke test: list cases",
            tool_calls=[{"name": "inbox.list_cases", "args": {}}],
            answer=None,
        )
    )
    print(f"smoke step reward={step.reward:.3f} done={step.done}")
    for tr in step.observation.tool_results[-1:]:
        print(f"{tr.name} ok={tr.ok}\n{str(tr.result)[:400]}")
finally:
    # async client backs this, close releases the HTTP pool
    if hasattr(env, "close"):
        env.close()

print("\nSmoke test passed. Environment is ready for training.")

## 4. Import Gemma 4 Training Utilities

Pull the env-wrapper class and reward functions from `support_ops_env.train_gemma4`. The class exposes 14 support-ops tools as Python methods; TRL introspects their docstrings + type hints to build the tool schema Gemma 4 sees in the chat template.

In [ ]:
from support_ops_env import get_gemma4_training_utils

g4 = get_gemma4_training_utils()
SYSTEM_PROMPT       = g4["SYSTEM_PROMPT"]
SupportOpsToolEnv   = g4["SupportOpsToolEnv"]
reward_total        = g4["reward_total"]
reward_investigation= g4["reward_investigation"]
reward_routing      = g4["reward_routing"]
reward_reply        = g4["reward_reply"]
reward_groundedness = g4["reward_groundedness"]

# Wire the env URL into the class. TRL instantiates it per-generation with no args.
SupportOpsToolEnv._env_url = ENV_URL
SupportOpsToolEnv._task_id = None  # None = first task; pin to e.g. 'c1_access_lockout' if you want fixed

print("System prompt (first 200 chars):")
print(SYSTEM_PROMPT[:200])
print("\nTools exposed by SupportOpsToolEnv:")
print([m for m in dir(SupportOpsToolEnv) if not m.startswith("_") and callable(getattr(SupportOpsToolEnv, m))])

## 5. Dataset + Trainer Setup

The dataset is just one seed user prompt per episode — TRL's `environment_factory` drives the rest of the rollout via tool calls.

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
import torch

# -------- 4-bit quantized base model (fits Gemma 4 E2B on T4 with LoRA) --------
quant_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"loading {MODEL_ID} in 4-bit ...")
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=quant_cfg,
    dtype=torch.bfloat16,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print(f"loaded. vram: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# -------- Dataset: one seed prompt per episode (env drives the rest) --------
dataset = Dataset.from_dict({
    "prompt": [
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": (
                "Triage and resolve every real case in the inbox. Investigate before "
                "you route. Draft one grounded reply. Call submit_resolution when done."
            )},
        ]
        for _ in range(NUM_EPISODES)
    ]
})

# -------- LoRA: train the language adapter only; skip the vision/audio towers --------
peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    bias="none", task_type="CAUSAL_LM",
    target_modules="all-linear",
    exclude_modules=["vision_tower", "multi_modal_projector", "audio_tower"],
)

# -------- GRPO trainer: pass the already-loaded model + processor's tokenizer --------
trainer = GRPOTrainer(
    model=base_model,
    processing_class=processor,
    train_dataset=dataset,
    reward_funcs=[reward_total, reward_investigation, reward_routing, reward_reply, reward_groundedness],
    peft_config=peft_config,
    args=GRPOConfig(
        chat_template_kwargs={"enable_thinking": ENABLE_THINKING},
        log_completions=True,
        logging_steps=2,
        num_completions_to_print=1,
        max_completion_length=MAX_COMPLETION_LEN,
        max_tool_calling_iterations=MAX_TOOL_ITERS,
        per_device_train_batch_size=NUM_GENERATIONS,
        steps_per_generation=1,
        num_generations=NUM_GENERATIONS,
        learning_rate=5e-6,
        gradient_accumulation_steps=4,
        max_steps=-1,
        report_to="none",
        save_strategy="no",
    ),
    environment_factory=SupportOpsToolEnv,
)

print("\nGRPOTrainer initialized")
print(f"vram after trainer init: {torch.cuda.memory_allocated()/1e9:.2f} GB / 15.8 GB total on T4")

## 6. Train

Each episode: TRL spawns a fresh `SupportOpsToolEnv` → Gemma 4 investigates via tool calls → drafts a reply → calls `submit_resolution` → grader scores → GRPO gradient update. Expect ~60-90 minutes on Colab T4 for 50 episodes.

In [ ]:
print(f"Starting GRPO training on {MODEL_ID}")
print(f"  Episodes    : {NUM_EPISODES}")
print(f"  Generations : {NUM_GENERATIONS}")
print(f"  Thinking    : {ENABLE_THINKING}")
print(f"  Environment : {ENV_URL}\n")

trainer.train()
print("\nTraining complete.")

## 7. Push LoRA to Hub (Optional)

Save the adapter and publish it so judges can pull the trained model.

In [ ]:
# Uncomment to push:
# trainer.push_to_hub(repo_id=HUB_REPO)
# print(f"Model pushed to https://huggingface.co/{HUB_REPO}")

## 8. Inference Sanity Check

Reload the trained LoRA adapter on top of the base model and run one fresh episode end-to-end to visually confirm the agent's behaviour improved.

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import PeftModel
import torch

processor = AutoProcessor.from_pretrained(MODEL_ID)
base = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype="auto", device_map="auto")
model = PeftModel.from_pretrained(base, trainer.args.output_dir)

sample_env = SupportOpsToolEnv()
print("Spawned a fresh env; you can now craft a prompt and call model.generate(...) manually.")
print("Keep this cell minimal on Colab T4 — the full tool-calling loop runs inside trainer.train().")